In [ ]:
from tqdm import tqdm
import time
import pandas as pd
import numpy as np
from pydantic import BaseModel, AliasChoices, Field, field_validator
from openai import OpenAI
import os
from collections import Counter
import json
import re
from typing import List, Optional, Literal, Union


Загрузка датасета и используемые модели

In [ ]:
df = pd.read_csv('')

In [ ]:
os.environ["VSELLM_API_KEY"] = ""

client = OpenAI(
    api_key=os.environ["VSELLM_API_KEY"],
    base_url="https://api.vsellm.ru/v1"
)

DEFAULT_MODEL = "openai/gpt-5.1"
DEFAULT_MODEL_1 = "qwen/qwen3.6-35b-a3b"

# DEFAULT_MODEL = "deepseek/deepseek-v3.2"
# DEFAULT_MODEL_1 = "qwen/qwen3.6-35b-a3b"

Baseline

In [ ]:
def safe_json_load(text: str):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError(f"No JSON found in: {text}")
        return json.loads(match.group(0))


In [ ]:
class SimpleAgentOutput(BaseModel):
    final_answer: str

SOLVER_PROMPT = """
You are a precise mathematical SOLVER agent.

You are given a mathematical problem.
Your goal is to compute the correct final answer accurately.

Instructions:
- Think carefully internally before answering
- Verify arithmetic and logical steps internally
- If calculations are complex, be extra careful
- Do not output reasoning or explanations
- Do not output intermediate steps
- Output ONLY a valid JSON object

Required JSON format:

{
  "final_answer": "answer"
}

IMPORTANT:
You MUST fill ALL required fields.
DO NOT omit or skip any field.
Even if you are uncertain, provide your best attempt for every field.

REQUIRED FIELDS:
- Answer (required, final answer)

Rules for final_answer:
- Must contain ONLY the final result
- No explanations
- No extra text
- No markdown
- No code blocks
- No units unless explicitly required
- Numbers must use digits only
- Lists must use comma-separated values

Examples:

Problem:
What is 2+2?

Output:
{
  "final_answer": "4"
}

IMPORTANT:
- Your entire response must be valid JSON
- The first character must be {
- The last character must be }
- Never include text before or after JSON
"""

class SimpleAgentOutput(BaseModel):
    final_answer: str


def run_simple_agent(question: str, k: int = 5):

    answers = []
    total_usage = {
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0
    }

    for _ in range(k):

        try:
            response = client.chat.completions.create(
                model=DEFAULT_MODEL,
                temperature=0.7,
                messages=[
                    {"role": "system", "content": SOLVER_PROMPT},
                    {"role": "user", "content": question}
                ]
            )

            content = response.choices[0].message.content

            data = json.loads(content)

            result = SimpleAgentOutput(**data)
            answers.append(result.final_answer)

            if response.usage:
                total_usage["prompt_tokens"] += response.usage.prompt_tokens
                total_usage["completion_tokens"] += response.usage.completion_tokens
                total_usage["total_tokens"] += response.usage.total_tokens

        except Exception as e:
            print("Error:", e)
            continue

    if len(answers) == 0:
        return SimpleAgentOutput(final_answer="no"), total_usage

    most_common_answer = Counter(answers).most_common(1)[0][0]

    return SimpleAgentOutput(final_answer=most_common_answer), total_usage


In [ ]:
pred_answer = []
correct_answer = []

total_time = 0
total_tokens = 0

for i in tqdm(range(len(df))):

    question = df.iloc[i]['problem']

    start = time.time()

    try:
        result, usage = run_simple_agent(question)

        answer = result.final_answer

        # tokens
        if usage:
            total_tokens += usage.get("total_tokens", 0)

    except Exception as e:
        answer = "no"
        usage = None
        print("ERROR:", e)

    total_time += time.time() - start

    pred_answer.append(answer)
    correct_answer.append(df.iloc[i]['answer'])

    print("Model answer:", answer)
    print("Real answer:", df.iloc[i]['answer'])

In [ ]:
correct = 0
total = len(pred_answer)

for pred, true in zip(pred_answer, correct_answer):


    pred_norm = str(pred).strip().lower()
    true_norm = str(true).strip().lower()

    if pred_norm == true_norm:
        correct += 1

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")
print(f"Avg time: {total_time/len(df):.4f}")
print(f"Total tokens: {total_tokens:.4f}")

In [ ]:
N_RUNS = 5

all_accuracies = []
all_avg_times = []
all_total_tokens = []

for run_idx in range(N_RUNS):

    print(f"\n===== RUN {run_idx + 1}/{N_RUNS} =====")

    pred_answer = []
    correct_answer = []

    total_time = 0
    total_tokens = 0

    for i in tqdm(range(len(df))):

        question = df.iloc[i]['problem']

        start = time.time()

        try:
            result, usage = run_simple_agent(question)

            answer = result.final_answer

            # токены
            if usage:
                total_tokens += usage.get("total_tokens", 0)

        except Exception as e:
            answer = "no"
            usage = None
            print("ERROR:", e)

        total_time += time.time() - start

        pred_answer.append(answer)
        correct_answer.append(df.iloc[i]['answer'])

        print("Model answer:", answer)
        print("Real answer:", df.iloc[i]['answer'])

    correct = 0
    total = len(pred_answer)

    for pred, true in zip(pred_answer, correct_answer):

        pred_norm = str(pred).strip().lower()
        true_norm = str(true).strip().lower()

        if pred_norm == true_norm:
            correct += 1

    accuracy = correct / total
    avg_time = total_time / len(df)

    # метрики
    all_accuracies.append(accuracy)
    all_avg_times.append(avg_time)
    all_total_tokens.append(total_tokens)

    print(f"\nRun {run_idx + 1} results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Avg time: {avg_time:.4f}")
    print(f"Total tokens: {total_tokens}")

In [ ]:
print(f"Mean Accuracy: {np.mean(all_accuracies):.4f}")
print(f"Std Accuracy: {np.std(all_accuracies):.4f}")
print(f"Mean Avg Time: {np.mean(all_avg_times):.4f}")
print(f"Std Avg Time: {np.std(all_avg_times):.4f}")
print(f"Mean Total Tokens: {np.mean(all_total_tokens):.2f}")
print(f"Std Total Tokens: {np.std(all_total_tokens):.2f}")

модуль планировщик + основной модуль + вычислительный модуль + модуль верификатор





In [ ]:
def extract_python(text: str):
    match = re.search(r"PYTHON:\s*(.*)", text, re.DOTALL)
    return match.group(1).strip() if match else None

def python_tool(code: str) -> str:
    try:
        local_vars = {}

        exec(
            code,
            {"__builtins__": {}},
            local_vars
        )

        return str(local_vars)

    except Exception as e:
        return f"Error: {e}"

In [ ]:
PLANNER_PROMPT = """
You are a PLANNER agent.

Your job:
1. Read the problem carefully
2. Break it into steps
3. Do NOT solve it
4. Output only a short step-by-step plan

IMPORTANT:
- reasoning must be under 100 words

Keep the plan concise.
"""

SOLVER_PROMPT = """
You are a mathematical SOLVER agent.

You receive:
- a problem
- optionally a plan

Your task:
1. Follow the plan if provided
2. Solve the problem step-by-step internally
3. Verify calculations carefully

You may use Python for calculations.

If needed, output:

PYTHON:
<code>

IMPORTANT:
- reasoning is INTERNAL
- reasoning must be under 500 words
- final_answer must contain ONLY the final result
- Use Python only for calculations
- Do NOT use multiple Python blocks
- Keep reasoning concise
- Output must be valid JSON OR single Python block
- do NOT put explanations inside final_answer

Use Python if:
- multiplication > 2 numbers
- combinatorics
- probability
- large integers

Return ONLY valid JSON.

The response MUST be a valid JSON object.

Example JSON:
{
  "reasoning": "step-by-step reasoning",
  "final_answer": "42"
}

Return ONLY valid JSON.

No extra text.
First character must be {
Last character must be }
No markdown
No code blocks
"""

VERIFIER_PROMPT = """
You are a mathematical VERIFIER agent.

You receive:
- a math problem
- solver reasoning
- a proposed final answer

Your task:
1. Check whether the reasoning is logically correct
2. Verify all calculations carefully
3. Detect arithmetic mistakes
4. Detect logical inconsistencies
5. Check whether the final_answer follows from the reasoning

IMPORTANT:
- Be skeptical and critical
- Do NOT trust the solver automatically
- Keep verification concise
- Do NOT solve the entire problem again unless necessary

Return ONLY valid JSON.

Possible outputs:

{
  "verdict": "CORRECT",
  "feedback": "brief explanation"
}

or

{
  "verdict": "INCORRECT",
  "feedback": "describe the mistake briefly"
}

If incorrect:
- identify exact mistake
- explain briefly why it's wrong
- suggest correction

Importance:
Feedback should be in one sentence, maximum 20 words.

"""



In [ ]:
#планер
def run_planner(question: str):

    response = client.chat.completions.create(
        model=DEFAULT_MODEL_1,
        temperature=0.4,
        max_tokens=500,
        messages=[
            {"role": "system", "content": PLANNER_PROMPT},
            {"role": "user", "content": question},
        ],
    )

    return response.choices[0].message.content, response.usage

In [ ]:
#решатель

class SolverOutput(BaseModel):

    reasoning: Optional[list[str]] = None
    final_answer: str

    @field_validator("reasoning", mode="before")
    @classmethod
    def normalize_reasoning(cls, v):

        if v is None:
            return []

        if isinstance(v, str):

            steps = re.split(r"\n?\d+\.\s*", v)

            steps = [s.strip() for s in steps if s.strip()]

            if steps:
                return steps

            return [v]

        return v

def run_solver(question: str, plan: str):

    messages = [
        {"role": "system", "content": SOLVER_PROMPT},
        {
            "role": "user",
            "content": f"Problem:\n{question}\n\nPlan:\n{plan}",
        },
    ]

    total_usage = None

    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        temperature=0.2,
        messages=messages,
    )

    total_usage = response.usage
    content = response.choices[0].message.content

    python_code = extract_python(content)

    if python_code:

        tool_result = python_tool(python_code)

        messages.append({"role": "assistant", "content": content})

        messages.append({
            "role": "user",
            "content": f"Internal computation result:\n{tool_result}"
        })

        final = client.chat.completions.create(
            model=DEFAULT_MODEL,
            temperature=0.2,
            messages=messages + [
                {
                    "role": "system",
                    "content": "Return ONLY JSON with reasoning and final_answer."
                }
            ],
        )

        if final.usage:
            total_usage.prompt_tokens += final.usage.prompt_tokens
            total_usage.completion_tokens += final.usage.completion_tokens
            total_usage.total_tokens += final.usage.total_tokens

        return SolverOutput(**safe_json_load(final.choices[0].message.content)), total_usage

    return SolverOutput(**safe_json_load(content)), total_usage

In [ ]:
#верификатор
class VerifierOutput(BaseModel):
    verdict: str
    feedback: str


def run_verifier(
    question: str,
    reasoning: str,
    final_answer: str,
):

    verify_input = f"""
Problem:
{question}

Solver reasoning:
{reasoning}

Solver final answer:
{final_answer}
"""

    response = client.chat.completions.create(
        model=DEFAULT_MODEL_1,
        temperature=0.0,
        max_tokens=200,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": VERIFIER_PROMPT},
            {"role": "user", "content": verify_input},
        ],
    )

    data = safe_json_load(response.choices[0].message.content)

    return VerifierOutput(**data), response.usage

In [ ]:
def run_multi_agent(question: str) -> dict:

    total_tokens = 0

    # STEP 1: planning
    plan, usage = run_planner(question)
    total_tokens += usage.total_tokens

    # STEP 2: first solve
    solver_result, usage = run_solver(question, plan)
    total_tokens += usage.total_tokens

    # STEP 3: verify
    verifier_result, usage = run_verifier(
        question=question,
        reasoning=solver_result.reasoning,
        final_answer=solver_result.final_answer,
    )
    total_tokens += usage.total_tokens

    # STEP 4: retry if incorrect
    if verifier_result.verdict == "INCORRECT":

        retry_plan = f"""
Original plan:
{plan}

Verifier feedback:
{verifier_result.feedback}

Solve again carefully.
"""

        solver_result, usage = run_solver(question, retry_plan)
        total_tokens += usage.total_tokens

        verifier_result, usage = run_verifier(
            question=question,
            reasoning=solver_result.reasoning,
            final_answer=solver_result.final_answer,
        )
        total_tokens += usage.total_tokens

    return {
    "final_answer": solver_result.final_answer,
    "question": question,

    "plan": str(plan),

    "solver_reasoning": solver_result.reasoning,
    "solver_answer": solver_result.final_answer,

    "verifier_verdict": verifier_result.verdict,
    "verifier_feedback": verifier_result.feedback,

    "total_tokens": total_tokens,
}

In [ ]:
result = run_multi_agent(df['problem'][0])

print(result)

In [ ]:
results = []

total_tokens = 0
total_time = 0

for i in tqdm(range(len(df))):

    question = df.iloc[i]['problem']
    true_answer = df.iloc[i]['answer']

    start = time.time()

    try:
        result = run_multi_agent(question)

        answer = result['final_answer']

        tokens = result.get("total_tokens", 0)

        total_tokens += tokens

    except Exception as e:

        answer = "no"
        tokens = 0

        result = {
            "error": str(e)
        }

        print("ERROR:", e)

    elapsed = time.time() - start

    total_time += elapsed

    is_correct = (
        str(answer).strip().lower()
        ==
        str(true_answer).strip().lower()
    )

    if not is_correct:

        results.append({

            "index": i,

            "question": question,

            "true_answer": true_answer,
            "pred_answer": answer,

            "correct": is_correct,

            "tokens": tokens,
            "time_sec": elapsed,

            "verdict": result.get("verdict"),
            "feedback": result.get("feedback"),

            "reasoning": result.get("solver_reasoning"),
            "plan": result.get("plan"),

            "error": result.get("error"),
        })

    print("Model answer:", answer)
    print("Real answer:", true_answer)

In [ ]:
#сборка текстовых ответов
results_df = pd.DataFrame(results)
results_df.to_csv("agent_results_medium.csv", index=False)

In [ ]:
pred_answer = []
correct_answer = []

total_tokens = 0
total_time = 0

for i in tqdm(range(len(df))):

    question = df.iloc[i]['problem']

    start = time.time()

    try:
        result = run_multi_agent(question)

        answer = result['final_answer']

        tokens = result.get("total_tokens", 0)

        total_tokens += tokens

    except Exception as e:

        answer = "no"
        tokens = 0

        print("ERROR:", e)

    elapsed = time.time() - start

    total_time += elapsed

    pred_answer.append(answer)
    correct_answer.append(df.iloc[i]['answer'])

    print("Model answer:", answer)
    print("Real answer:", df.iloc[i]['answer'])

In [ ]:
#независимые запуски
N_RUNS = 5

all_accuracies = []
all_avg_times = []
all_total_tokens = []

for run_idx in range(N_RUNS):

    print(f"\n===== RUN {run_idx + 1}/{N_RUNS} =====")

    pred_answer = []
    correct_answer = []

    total_tokens = 0
    total_time = 0

    for i in tqdm(range(len(df))):

        question = df.iloc[i]['problem']

        start = time.time()

        try:
            result = run_multi_agent(question)

            answer = result['final_answer']

            tokens = result.get("total_tokens", 0)

            total_tokens += tokens

        except Exception as e:

            answer = "no"
            tokens = 0

            print("ERROR:", e)

        elapsed = time.time() - start

        total_time += elapsed

        pred_answer.append(answer)
        correct_answer.append(df.iloc[i]['answer'])

        print("Model answer:", answer)
        print("Real answer:", df.iloc[i]['answer'])

    correct = 0
    total = len(pred_answer)

    for pred, true in zip(pred_answer, correct_answer):

        pred_norm = str(pred).strip().lower()
        true_norm = str(true).strip().lower()

        if pred_norm == true_norm:
            correct += 1

    accuracy = correct / total
    avg_time = total_time / len(df)

    all_accuracies.append(accuracy)
    all_avg_times.append(avg_time)
    all_total_tokens.append(total_tokens)

    print(f"\nRun {run_idx + 1} results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Avg time: {avg_time:.4f}")
    print(f"Total tokens: {total_tokens}")


In [ ]:
print(f"Mean Accuracy: {np.mean(all_accuracies):.4f}")
print(f"Std Accuracy: {np.std(all_accuracies):.4f}")
print(f"Mean Avg Time: {np.mean(all_avg_times):.4f}")
print(f"Std Avg Time: {np.std(all_avg_times):.4f}")
print(f"Mean Total Tokens: {np.mean(all_total_tokens):.2f}")
print(f"Std Avg Time: {np.std(all_avg_times):.4f}")


основной модуль + вычислительный модуль

In [ ]:
SOLVER_PROMPT = """
You are a mathematical SOLVER agent.

You receive:
- a problem

Your task:
1. Solve the problem step-by-step internally
2. Verify calculations carefully

You may use Python for calculations.

If needed, output:

PYTHON:
<code>

IMPORTANT:
- reasoning is INTERNAL
- reasoning must be under 500 words
- final_answer must contain ONLY the final result
- Use Python only for calculations
- Do NOT use multiple Python blocks
- Keep reasoning concise
- Output must be valid JSON OR single Python block
- do NOT put explanations inside final_answer

Use Python if:
- multiplication > 2 numbers
- combinatorics
- probability
- large integers

Return ONLY valid JSON.

The response MUST be a valid JSON object.

Example JSON:
{
  "reasoning": "step-by-step reasoning",
  "final_answer": "42"
}

Return ONLY valid JSON.

No extra text.
First character must be {
Last character must be }
No markdown
No code blocks
"""

In [ ]:
class SolverOutput(BaseModel):
    reasoning: str
    final_answer: str

def run_solver(question: str):

    messages = [
        {"role": "system", "content": SOLVER_PROMPT},
        {
            "role": "user",
            "content": f"Problem:\n{question}",
        },
    ]

    total_tokens = 0

    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        temperature=0.2,
        messages=messages,
    )

    if response.usage:
        total_tokens += response.usage.total_tokens

    content = response.choices[0].message.content

    python_code = extract_python(content)

    # CASE 1: Python needed
    if python_code:

        tool_result = python_tool(python_code)

        messages.append({"role": "assistant", "content": content})

        messages.append({
            "role": "user",
            "content": f"Internal computation result:\n{tool_result}"
        })

        final = client.chat.completions.create(
            model=DEFAULT_MODEL,
            temperature=0.2,
            messages=messages + [
                {
                    "role": "system",
                    "content": "Return ONLY JSON with reasoning and final_answer."
                }
            ],
        )

        if final.usage:
            total_tokens += final.usage.total_tokens

        return (
            SolverOutput(**safe_json_load(final.choices[0].message.content)),
            total_tokens
        )

    return (
        SolverOutput(**safe_json_load(content)), total_tokens)

In [ ]:
def run_python_agent(question: str) -> dict:

    total_tokens = 0

    solver_result, solver_tokens = run_solver(question)

    total_tokens += solver_tokens

    return {
        "final_answer": solver_result.final_answer,
        "total_tokens": total_tokens
    }

In [ ]:
pred_answer = []
correct_answer = []

total_tokens = 0
total_time = 0

for i in tqdm(range(len(df))):

    question = df.iloc[i]['problem']

    start = time.time()

    try:
        result = run_python_agent(question)

        answer = result["final_answer"]

        # tokens
        total_tokens += result.get("total_tokens", 0)

    except Exception as e:
        answer = "no"
        print("ERROR:", e)

    total_time += time.time() - start

    pred_answer.append(answer)
    correct_answer.append(df.iloc[i]['answer'])

    print("Model answer:", answer)
    print("Real answer:", df.iloc[i]['answer'])

In [ ]:
#независимые запуски для получение усредненных метрик
correct = 0
total = len(pred_answer)

for pred, true in zip(pred_answer, correct_answer):

    pred_norm = str(pred).strip().lower()
    true_norm = str(true).strip().lower()

    if pred_norm == true_norm:
        correct += 1

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")
print(f"Avg time: {total_time/len(df):.4f}")
print(f"Total tokens: {total_tokens:.4f}")

In [ ]:
#независимые запуски для получения усредненных метрик и стандартного отклонения
N_RUNS = 5

all_accuracies = []
all_avg_times = []
all_total_tokens = []

for run_idx in range(N_RUNS):

    print(f"\n===== RUN {run_idx + 1}/{N_RUNS} =====")

    pred_answer = []
    correct_answer = []

    total_tokens = 0
    total_time = 0

    for i in tqdm(range(len(df))):

        question = df.iloc[i]['problem']

        start = time.time()

        try:
            result = run_python_agent(question)

            answer = result["final_answer"]

            # токены
            total_tokens += result.get("total_tokens", 0)

        except Exception as e:
            answer = "no"
            print("ERROR:", e)

        total_time += time.time() - start

        pred_answer.append(answer)
        correct_answer.append(df.iloc[i]['answer'])

        print("Model answer:", answer)
        print("Real answer:", df.iloc[i]['answer'])

    correct = 0
    total = len(pred_answer)

    for pred, true in zip(pred_answer, correct_answer):

        pred_norm = str(pred).strip().lower()
        true_norm = str(true).strip().lower()

        if pred_norm == true_norm:
            correct += 1

    accuracy = correct / total
    avg_time = total_time / len(df)

    all_accuracies.append(accuracy)
    all_avg_times.append(avg_time)
    all_total_tokens.append(total_tokens)

    print(f"\nRun {run_idx + 1} results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Avg time: {avg_time:.4f}")
    print(f"Total tokens: {total_tokens}")

In [ ]:
print(f"Mean Accuracy: {np.mean(all_accuracies):.4f}")
print(f"Std Accuracy: {np.std(all_accuracies):.4f}")
print(f"Mean Avg Time: {np.mean(all_avg_times):.4f}")
print(f"Std Avg Time: {np.std(all_avg_times):.4f}")
print(f"Mean Total Tokens: {np.mean(all_total_tokens):.2f}")
print(f"Std Avg Time: {np.std(all_avg_times):.4f}")


основной модуль + вычислительный модуль + модуль верификатор

In [ ]:
SOLVER_PROMPT = """
You are a mathematical SOLVER agent.

You receive:
- a problem

Your task:
1. Solve the problem step-by-step internally
2. Verify calculations carefully

You may use Python for calculations.

If needed, output:

PYTHON:
<code>

IMPORTANT:
- reasoning is INTERNAL
- reasoning must be under 500 words
- final_answer must contain ONLY the final result
- Use Python only for calculations
- Do NOT use multiple Python blocks
- Keep reasoning concise
- Output must be valid JSON OR single Python block
- do NOT put explanations inside final_answer

Use Python if:
- multiplication > 2 numbers
- combinatorics
- probability
- large integers

Return ONLY valid JSON.

The response MUST be a valid JSON object.

Example JSON:
{
  "reasoning": "step-by-step reasoning",
  "final_answer": "42"
}

Return ONLY valid JSON.

No extra text.
First character must be {
Last character must be }
No markdown
No code blocks
"""

In [ ]:
class SolverOutput(BaseModel):
    reasoning: str
    final_answer: str


def run_solver(question: str):

    messages = [
        {"role": "system", "content": SOLVER_PROMPT},
        {
            "role": "user",
            "content": f"Problem:\n{question}",
        },
    ]

    total_tokens = 0

    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        temperature=0.2,
        messages=messages,
    )

    if response.usage:
        total_tokens += response.usage.total_tokens

    content = response.choices[0].message.content

    python_code = extract_python(content)

    if python_code:

        tool_result = python_tool(python_code)

        messages.append({"role": "assistant", "content": content})

        messages.append({
            "role": "user",
            "content": f"Internal computation result:\n{tool_result}"
        })

        final = client.chat.completions.create(
            model=DEFAULT_MODEL,
            temperature=0.2,
            messages=messages + [
                {
                    "role": "system",
                    "content": "Return ONLY JSON with reasoning and final_answer."
                }
            ],
        )

        if final.usage:
            total_tokens += final.usage.total_tokens

        return (
            SolverOutput(**safe_json_load(final.choices[0].message.content)),
            total_tokens
        )

    return (
        SolverOutput(**safe_json_load(content)),
        total_tokens
    )

In [ ]:
class VerifierOutput(BaseModel):
    verdict: str
    feedback: str


def run_verifier(
    question: str,
    reasoning: str,
    final_answer: str,
) -> VerifierOutput:

    verify_input = f"""
Problem:
{question}

Solver reasoning:
{reasoning}

Solver final answer:
{final_answer}
"""

    response = client.chat.completions.create(
        model=DEFAULT_MODEL_1,
        temperature=0.0,
        max_tokens=1200,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": VERIFIER_PROMPT
            },
            {
                "role": "user",
                "content": verify_input
            },
        ],
    )

    total_tokens = 0

    if response.usage:
        total_tokens += response.usage.total_tokens

    content = response.choices[0].message.content

    data = safe_json_load(content)

    return (
        VerifierOutput(**data),
        total_tokens
    )

In [ ]:
def run_verifier_agent(question: str) -> dict:

    total_tokens = 0

    solver_result, solver_tokens = run_solver(question)
    total_tokens += solver_tokens

    verifier_result, verifier_tokens = run_verifier(
        question=question,
        reasoning=solver_result.reasoning,
        final_answer=solver_result.final_answer,
    )

    total_tokens += verifier_tokens

    if verifier_result.verdict == "INCORRECT":

        question = f"""
Question:
{question}

Verifier feedback:
{verifier_result.feedback}

Solve again carefully.
"""

        solver_result, solver_tokens = run_solver(question)
        total_tokens += solver_tokens

        verifier_result, verifier_tokens = run_verifier(
            question=question,
            reasoning=solver_result.reasoning,
            final_answer=solver_result.final_answer,
        )

        total_tokens += verifier_tokens

    return {
        "final_answer": solver_result.final_answer,
        "verdict": verifier_result.verdict,
        "feedback": verifier_result.feedback,
        "total_tokens": total_tokens,
    }

In [ ]:
from tqdm import tqdm
import time

pred_answer = []
correct_answer = []

total_tokens = 0
total_time = 0

for i in tqdm(range(len(df))):

    question = df.iloc[i]['problem']

    start = time.time()

    try:
        result = run_verifier_agent(question)

        answer = result['final_answer']

        # token count
        total_tokens += result.get("total_tokens", 0)

    except Exception as e:
        answer = "no"
        print("ERROR:", e)

    total_time += time.time() - start

    pred_answer.append(answer)
    correct_answer.append(df.iloc[i]['answer'])

    print('Model answer:', answer)
    print('Real answer:', df.iloc[i]['answer'])

In [ ]:
correct = 0
total = len(pred_answer)

for pred, true in zip(pred_answer, correct_answer):

    pred_norm = str(pred).strip().lower()
    true_norm = str(true).strip().lower()

    if pred_norm == true_norm:
        correct += 1

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")
print(f"Avg time: {total_time/len(df):.4f}")
print(f"Total tokens: {total_tokens:.4f}")

In [ ]:
# независимые запуски
N_RUNS = 3

all_accuracies = []
all_avg_times = []
all_total_tokens = []

for run_idx in range(N_RUNS):

    print(f"\n===== RUN {run_idx + 1}/{N_RUNS} =====")

    pred_answer = []
    correct_answer = []

    total_tokens = 0
    total_time = 0

    for i in tqdm(range(len(df))):

        question = df.iloc[i]['problem']

        start = time.time()

        try:
            result = run_multi_agent(question)

            answer = result['final_answer']

            total_tokens += result.get("total_tokens", 0)

        except Exception as e:
            answer = "no"
            print("ERROR:", e)

        total_time += time.time() - start

        pred_answer.append(answer)
        correct_answer.append(df.iloc[i]['answer'])

        print('Model answer:', answer)
        print('Real answer:', df.iloc[i]['answer'])

    correct = 0
    total = len(pred_answer)

    for pred, true in zip(pred_answer, correct_answer):

        pred_norm = str(pred).strip().lower()
        true_norm = str(true).strip().lower()

        if pred_norm == true_norm:
            correct += 1

    accuracy = correct / total
    avg_time = total_time / len(df)

    # сохраняем результаты запуска
    all_accuracies.append(accuracy)
    all_avg_times.append(avg_time)
    all_total_tokens.append(total_tokens)

    print(f"\nRun {run_idx + 1} results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Avg time: {avg_time:.4f}")
    print(f"Total tokens: {total_tokens}")

In [ ]:
print(f"Mean Accuracy: {np.mean(all_accuracies):.4f}")
print(f"Std Accuracy: {np.std(all_accuracies):.4f}")
print(f"Mean Avg Time: {np.mean(all_avg_times):.4f}")
print(f"Std Avg Time: {np.std(all_avg_times):.4f}")
print(f"Mean Total Tokens: {np.mean(all_total_tokens):.2f}")
print(f"Std Total Tokens: {np.std(all_total_tokens):.2f}")

модуль планировщик + основной модуль + вычислительный модуль

In [ ]:
def run_planner(question: str):

    response = client.chat.completions.create(
        model=DEFAULT_MODEL_1,
        temperature=0.4,
        max_tokens=500,
        messages=[
            {"role": "system", "content": PLANNER_PROMPT},
            {"role": "user", "content": question},
        ],
    )

    return response.choices[0].message.content, response.usage

In [ ]:
class SolverOutput(BaseModel):
    reasoning: str
    final_answer: str

def run_solver(question: str, plan: str):

    messages = [
        {"role": "system", "content": SOLVER_PROMPT},
        {
            "role": "user",
            "content": f"Problem:\n{question}\n\nPlan:\n{plan}",
        },
    ]

    total_tokens = 0

    # STEP 1: model decides
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        temperature=0.2,
        messages=messages,
    )

    # token count
    if response.usage:
        total_tokens += response.usage.total_tokens

    content = response.choices[0].message.content

    python_code = extract_python(content)

    # CASE 1: Python needed
    if python_code:

        tool_result = python_tool(python_code)

        # inject result back silently
        messages.append({"role": "assistant", "content": content})

        messages.append({
            "role": "user",
            "content": f"Internal computation result:\n{tool_result}"
        })

        # final completion (NO python visible)
        final = client.chat.completions.create(
            model=DEFAULT_MODEL,
            temperature=0.2,
            messages=messages + [
                {
                    "role": "system",
                    "content": "Return ONLY JSON with reasoning and final_answer."
                }
            ],
        )

        # token count
        if final.usage:
            total_tokens += final.usage.total_tokens

        return (
            SolverOutput(**safe_json_load(final.choices[0].message.content)),
            total_tokens
        )

    # CASE 2: no Python
    return (
        SolverOutput(**safe_json_load(content)),
        total_tokens
    )

In [ ]:
def run_planner_agent(question: str) -> dict:

    total_tokens = 0

    plan, planner_tokens = run_planner(question)
    total_tokens += planner_tokens.total_tokens

    solver_result, solver_tokens = run_solver(question, plan)
    total_tokens += solver_tokens

    return {
        "final_answer": solver_result.final_answer,
        "total_tokens": total_tokens
    }

In [ ]:
pred_answer = []
correct_answer = []

total_tokens = 0
total_time = 0

for i in tqdm(range(len(df))):

    question = df.iloc[i]['problem']

    start = time.time()

    try:
        result = run_planner_agent(question)

        answer = result['final_answer']

        total_tokens += result.get("total_tokens", 0)

    except Exception as e:
        answer = "no"
        print("ERROR:", e)

    total_time += time.time() - start

    pred_answer.append(answer)
    correct_answer.append(df.iloc[i]['answer'])

    print('Model answer:', answer)
    print('Real answer:', df.iloc[i]['answer'])

In [ ]:
correct = 0
total = len(pred_answer)

for pred, true in zip(pred_answer, correct_answer):

    pred_norm = str(pred).strip().lower()
    true_norm = str(true).strip().lower()

    if pred_norm == true_norm:
        correct += 1

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")
print(f"Avg time: {total_time/len(df):.4f}")
print(f"Total tokens: {total_tokens:.4f}")

In [ ]:
# независимые прогоны
N_RUNS = 3

all_accuracies = []
all_avg_times = []
all_total_tokens = []

for run_idx in range(N_RUNS):

    print(f"\n===== RUN {run_idx + 1}/{N_RUNS} =====")

    pred_answer = []
    correct_answer = []

    total_tokens = 0
    total_time = 0

    # цикл по датасету
    for i in tqdm(range(len(df))):

        question = df.iloc[i]['problem']

        start = time.time()

        try:
            result = run_multi_agent(question)

            answer = result['final_answer']

            total_tokens += result.get("total_tokens", 0)

        except Exception as e:
            answer = "no"
            print("ERROR:", e)

        total_time += time.time() - start

        pred_answer.append(answer)
        correct_answer.append(df.iloc[i]['answer'])

        print('Model answer:', answer)
        print('Real answer:', df.iloc[i]['answer'])

    correct = 0
    total = len(pred_answer)

    for pred, true in zip(pred_answer, correct_answer):

        pred_norm = str(pred).strip().lower()
        true_norm = str(true).strip().lower()

        if pred_norm == true_norm:
            correct += 1

    accuracy = correct / total
    avg_time = total_time / len(df)

    all_accuracies.append(accuracy)
    all_avg_times.append(avg_time)
    all_total_tokens.append(total_tokens)

    print(f"\nRun {run_idx + 1} Results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Avg time: {avg_time:.4f}")
    print(f"Total tokens: {total_tokens}")

In [ ]:
print(f"Mean Accuracy: {np.mean(all_accuracies):.4f}")
print(f"Std Accuracy: {np.std(all_accuracies):.4f}")
print(f"Mean Avg Time: {np.mean(all_avg_times):.4f}")
print(f"Std Avg Time: {np.std(all_avg_times):.4f}")
print(f"Mean Total Tokens: {np.mean(all_total_tokens):.2f}")
print(f"Std Avg Time: {np.std(all_avg_times):.4f}")

финальная архитектура

In [ ]:
FAST_MODEL = DEFAULT_MODEL
PLANNER_MODEL = DEFAULT_MODEL_1
SOLVER_MODEL = DEFAULT_MODEL
CRITIC_MODEL = DEFAULT_MODEL_1
VERIFIER_MODEL = DEFAULT_MODEL_1
REFINER_MODEL = DEFAULT_MODEL

In [ ]:
DIFFICULTY_PROMPT = """
You are a difficulty classifier.

Classify the user's problem into one category:

- EASY
- MEDIUM
- HARD

Guidelines:

EASY:
- direct factual question
- short reasoning
- simple arithmetic
- no decomposition needed

MEDIUM:
- multiple reasoning steps
- moderate planning
- possible calculations
- moderate ambiguity

HARD:
- long reasoning chains
- decomposition required
- high chance of reasoning failure
- tool usage likely needed
- verification important

Rules:
- Think carefully before classifying.
- Prefer HARD if uncertain.
- Return concise output only.
- Do not explain reasoning.
- Keep internal reasoning under 30 words.

Return valid JSON only.
"""


PLANNER_PROMPT = """
You are a JSON generator.

DO NOT REASON.
DO NOT EXPLAIN.
DO NOT THINK OUT LOUD.

Return ONLY valid JSON.

Format:

{
  "approach": "short phrase",
  "steps": [
    "max 3 words",
    "max 3 words"
  ]
}

Rules:
- exactly 3 steps
- each step ≤ 7 words
- no punctuation in steps
- no explanations
"""

SOLVER_PROMPT = """
You are an expert problem solver.

Your goal:
Find a correct and efficient solution.

IMPORTANT:
You MUST fill ALL required fields.
DO NOT omit or skip any field.
Even if you are uncertain, provide your best attempt for every field.

REQUIRED FIELDS:
- Reasoning (required, list of short reasoning steps)
- Answer (required, final answer)
- Confidence (required, float from 0 to 1)

The output is INVALID if any required field is missing.

Reasoning Rules:
- Think step by step.
- Verify intermediate conclusions internally; don't infer them.
- Avoid unsubstantiated assumptions.
- Don't describe your reasoning in detail; describe the key steps.
- Don't write out formulas in their entirety; reference them at most.
- Prefer precise reasoning to verbosity.
- Use tools for arithmetic calculations, symbolic manipulations, or large calculations.
- Never falsify calculations.

Using Tools:
If a tool is required, return a tool call.

Output Limits:
- Maximum 8 steps Reasoning.
- Each reasoning step is a maximum of 25 words.
- The total length of the reasoning should not exceed 180 words.
- The final answer must be a single number.
- The maximum number of reasoning steps in the answer is 4700.
- Confidence should reflect uncertainty, realistically.

Important:
- A concise summary is preferable.

- Precision is more important than style.

- Do not repeat the plan.

Return only valid JSON.
"""


CRITIC_PROMPT = """
You are a strict critic of reasoning.

Your task:
Find errors and improve the solution.

Check:
- logical errors
- arithmetic errors
- unfounded assumptions
- contradictions

Rules:
- Maximum 4 questions
- Each question less than 15 words
- Be concise
- Each discovered inaccuracy must be described as briefly as possible for further transfer to the next model. Each question less than 20 words
- It is extremely important to keep the number of tokens under 2700
- Do not write out the formulas in full, only reference them

MAIN RULE:
- Maximum number of tokens 2700

Return JSON EXACTLY:

{
"issues": [
"issue 1"
],

"improved_reasoning": [
"step 1"
],

"improved_answer": "answer",
"confidence": 0.85
}

Return only valid JSON.
"""

VERIFIER_PROMPT = """
You are a strict verifier.

YOU MUST return ALL fields.

Return EXACT JSON:

{
"score": 0.0-1.0,
"verdict": "CORRECT or INCORRECT",
"feedback": "short text"
}

RULES:
- You MUST include a verdict
- NO missing fields
- NO additional fields
- NO explanations outside the JSON
- Return ONLY valid JSON

Important:
- Feedback must be short. You don't need to describe all the errors, just provide brief feedback in no more than 50 words. 3 sentences maximum
- You must keep within the maximum number of tokens: 970

MAIN RULE:
- The maximum number of tokens is 970

Return only valid JSON JSON.
"""


REFINER_PROMPT = """
You are a reasoning refiner.

Your task:
Improve the candidate using critique feedback.

Requirements:
- Fix identified mistakes.
- Preserve correct reasoning.
- Increase precision and clarity.
- Avoid introducing new assumptions.

Rules:
- Maximum 8 reasoning steps.
- Each step maximum 25 words.
- Total reasoning under 180 words.
- Final maximum answer 40 words.
- Be concise and correct.

MAIN RULE:
- Maximum number of tokens 2700

Return valid JSON only.
"""

In [ ]:
class DifficultyOutput(BaseModel):

    level: Literal["EASY", "MEDIUM", "HARD"] = Field(
        validation_alias=AliasChoices(
            "level",
            "category",
            "difficulty",
        )
    )

class Plan(BaseModel):
    approach: str
    steps: List[str]


class ToolCall(BaseModel):
    tool: str
    code: str


class SolverCandidate(BaseModel):

    reasoning: list[str] = Field(
        validation_alias=AliasChoices(
            "reasoning",
            "Reasoning",
        )
    )

    answer: Union[str, int, float] = Field(
        validation_alias=AliasChoices(
            "answer",
            "Answer",
            "final_answer",
            "result",
        )
    )

    confidence: float = Field(
        validation_alias=AliasChoices(
            "confidence",
            "Confidence",
            "score",
        )
    )

    tool_call: Optional[dict] = None


class Critique(BaseModel):
    issues: List[str]
    improved_reasoning: List[str]
    improved_answer: str
    confidence: float


class Verification(BaseModel):
    score: float
    verdict: Literal["CORRECT", "INCORRECT"]
    feedback: str

In [ ]:
def extract_json(text):

    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match:
        raise ValueError("No JSON found")

    return match.group(0)

def structured_completion(
    model,
    messages,
    schema,
    max_tokens,
    temperature=0.2,
):

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        reasoning_effort="low",
        response_format={"type": "json_object"},
    )

    total_tokens = 0

    # token count
    if response.usage:
        total_tokens += response.usage.total_tokens

    content = response.choices[0].message.content

    try:

        data = json.loads(content)

    except Exception:

        print("\nRAW OUTPUT:\n")
        print(content)

        # try extracting json
        extracted = extract_json(content)

        data = json.loads(extracted)

    return (
        schema(**data),
        total_tokens
    )

In [ ]:
#классификатор задач
#если задача легкая будем решать проще
def estimate_difficulty(question: str):

    difficulty_result, total_tokens = structured_completion(
        model=QWEN_MODEL,
        messages=[
            {
                "role": "system",
                "content": DIFFICULTY_PROMPT,
            },
            {
                "role": "user",
                "content": question,
            },
        ],
        schema=DifficultyOutput,
        max_tokens=20,
        temperature=0,
    )

    return difficulty_result, total_tokens

In [ ]:
def generate_plans(question: str, k=2):

    plans = []
    total_tokens = 0

    for _ in range(k):

        plan, tokens = structured_completion(
            model=PLANNER_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": """
You are an expert planner.

Create a SHORT reasoning plan.

Rules:
- Maximum 4 steps
- Each step under 12 words
- Be concise

Return JSON EXACTLY:

{
  "approach": "short description",
  "steps": [
    "step 1",
    "step 2"
  ]
}

Return valid JSON only.
"""
                },
                {
                    "role": "user",
                    "content": question,
                },
            ],
            schema=Plan,
            max_tokens=500,
            temperature=0.2,
        )

        total_tokens += tokens

        plans.append(plan)

    return plans, total_tokens

In [ ]:
def run_tool(tool_call: ToolCall):

    if tool_call.tool == "python":

        return python_tool(tool_call.code)

    elif tool_call.tool == "calculator":

        return eval(tool_call.code)

    else:
        raise ValueError("Unknown tool")

In [ ]:
def solve_once(question, plan, difficulty):

    if difficulty == "EASY":
        max_tokens = 150

    elif difficulty == "MEDIUM":
        max_tokens = 5000

    else:
        max_tokens = 5000

    messages = [
        {
            "role": "system",
            "content": SOLVER_PROMPT,
        },
        {
            "role": "user",
            "content": f"""
Question:
{question}

Plan:
{plan.model_dump_json()}
"""
        }
    ]

    candidate = None
    total_tokens = 0

    for _ in range(3):

        candidate, tokens = structured_completion(
            model=SOLVER_MODEL,
            messages=messages,
            schema=SolverCandidate,
            max_tokens=max_tokens,
            temperature=0.2,
        )

        total_tokens += tokens

        # =================================================
        # TOOL USE
        # =================================================

        if candidate.tool_call:

            tool_result = run_tool(
                candidate.tool_call
            )

            messages.append({
                "role": "assistant",
                "content": candidate.model_dump_json(),
            })

            messages.append({
                "role": "tool",
                "content": str(tool_result),
            })

            continue

        # no tool needed
        return candidate, total_tokens

    return candidate, total_tokens

In [ ]:
def generate_candidates(question, plans, level, n_per_plan=3):

    candidates = []
    total_tokens = 0

    for plan in plans:

        for _ in range(n_per_plan):

            candidate, tokens = solve_once(question, plan, level)

            total_tokens += tokens

            candidates.append(candidate)

    return candidates, total_tokens

In [ ]:
def critique_candidate(question, candidate):

    critique_result, total_tokens = structured_completion(
        model=CRITIC_MODEL,
        messages=[
            {
                "role": "system",
                "content": CRITIC_PROMPT,
            },
            {
                "role": "user",
                "content": f"""
Question:
{question}

Candidate:
{candidate.model_dump_json()}
"""
            }
        ],
        schema=Critique,
        max_tokens=3000,
        temperature=0.1,
    )

    return critique_result, total_tokens

In [ ]:
def reflect_and_refine(question, candidate, rounds=2):

    current = candidate
    total_tokens = 0

    for _ in range(rounds):

        critique, tokens = critique_candidate(question, current)

        total_tokens += tokens

        current = SolverCandidate(
            reasoning=critique.improved_reasoning,
            answer=critique.improved_answer,
            confidence=critique.confidence,
        )

    return current, total_tokens

In [ ]:
def verify_candidate(question, candidate):

    verification_result, total_tokens = structured_completion(
        model=VERIFIER_MODEL,
        messages=[
            {
                "role": "system",
                "content": VERIFIER_PROMPT,
            },
            {
                "role": "user",
                "content": f"""
Question:
{question}

Candidate:
{candidate.model_dump_json()}
"""
            }
        ],
        schema=Verification,
        max_tokens=1000,
        temperature=0,
    )

    return verification_result, total_tokens

In [ ]:
def rank_candidates(candidates, verifications):

    scored = []

    for cand, ver in zip(candidates, verifications):

        score = ver.score * 0.7 + cand.confidence * 0.3

        scored.append((score, cand, ver))

    return sorted(scored, key=lambda x: x[0], reverse=True)

In [ ]:
from collections import Counter

def normalize_answer(answer):

    if answer is None:
        return "none"

    answer = str(answer).strip().lower()

    if answer.endswith(".0"):
        answer = answer[:-2]

    return answer

def majority_vote(candidates):

    answers = [normalize_answer(c.answer) for c in candidates]

    counter = Counter(answers)

    return counter.most_common(1)[0][0]

In [ ]:
def run_reasoning_system(question):

    total_tokens = 0

    # DIFFICULTY

    difficulty, tokens = estimate_difficulty(question)

    total_tokens += tokens

    level = difficulty.level

    print(f"\nDifficulty: {level}")

    # EASY

    if level == "EASY":

        print("Planning...")

        plans, tokens = generate_plans(
            question,
            k=1,
        )

        total_tokens += tokens

        print("Solving...")

        candidate, tokens = solve_once(
            question,
            plans[0],
            level,
        )

        total_tokens += tokens

        return {
            "final_answer": candidate.answer,
            "total_tokens": total_tokens,
        }

    # MEDIUM / HARD CONFIG

    if level == "MEDIUM":

        k_plans = 2
        n_per_plan = 1

    else:

        k_plans = 2
        n_per_plan = 2

    # PLANNING

    print("Planning...")

    plans, tokens = generate_plans(
        question,
        k=k_plans,
    )

    total_tokens += tokens

    # SOLVING

    print("Solving...")

    candidates, tokens = generate_candidates(
        question,
        plans,
        level,
        n_per_plan=n_per_plan
    )

    total_tokens += tokens

    # REFLECTION

    improved_candidates = []

    if level == "HARD":

        print("Refining...")

        for candidate in candidates:

            refined, tokens = reflect_and_refine(
                question,
                candidate,
            )

            total_tokens += tokens

            improved_candidates.append(refined)

    else:

        improved_candidates = candidates

    # VERIFICATION

    print("Verifying...")

    verifications = []

    for candidate in improved_candidates:

        verification, tokens = verify_candidate(
            question,
            candidate,
        )

        total_tokens += tokens

        verifications.append(verification)

    # RANKING

    ranked = rank_candidates(
        improved_candidates,
        verifications,
    )

    # VOTING

    top_candidates = [
        x[1]
        for x in ranked[:3]
    ]

    final_answer = majority_vote(
        top_candidates,
    )

    return {
        "final_answer": final_answer,
        "total_tokens": total_tokens,
    }

In [ ]:
import json

run_reasoning_system(df['problem'][0])

In [ ]:
pred_answer = []
correct_answer = []
errors = []

total_tokens = 0
total_time = 0

for i in tqdm(range(10)):

    question = df["problem"][i]
    gt_answer = df["answer"][i]

    start_time = time.time()

    try:

        response = run_reasoning_system(question)

        result = response["final_answer"]

        # token usage
        tokens = response.get("total_tokens", 0)
        total_tokens += tokens

        result = normalize_answer(result)
        gt_answer_norm = normalize_answer(gt_answer)

    except Exception as e:

        print(f"\nERROR at index {i}: {e}")

        result = "ERROR"
        gt_answer_norm = normalize_answer(gt_answer)

        errors.append({
            "index": i,
            "error": str(e),
        })

    elapsed = time.time() - start_time
    total_time += elapsed

    pred_answer.append(result)
    correct_answer.append(gt_answer_norm)

    print("Predicted:", result)
    print("Ground Truth:", gt_answer_norm)

In [ ]:
correct = 0
total = len(pred_answer)

for pred, true in zip(pred_answer, correct_answer):

    pred_norm = str(pred).strip().lower()
    true_norm = str(true).strip().lower()

    if pred_norm == true_norm:
        correct += 1

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")
print(f"Avg time: {total_time/len(df):.4f}")
print(f"Total tokens: {total_tokens:.4f}")

In [ ]:
# независимые запуски
N_RUNS = 5

all_accuracies = []
all_avg_times = []
all_total_tokens = []

for run_idx in range(N_RUNS):

    print(f"\n===== RUN {run_idx + 1}/{N_RUNS} =====")

    pred_answer = []
    correct_answer = []
    errors = []

    total_tokens = 0
    total_time = 0

    for i in tqdm(range(10)):

        question = df["problem"][i]
        gt_answer = df["answer"][i]

        start_time = time.time()

        try:

            response = run_reasoning_system(question)

            result = response["final_answer"]

            # token usage
            tokens = response.get("total_tokens", 0)
            total_tokens += tokens

            result = normalize_answer(result)
            gt_answer_norm = normalize_answer(gt_answer)

        except Exception as e:

            print(f"\nERROR at index {i}: {e}")

            result = "ERROR"
            gt_answer_norm = normalize_answer(gt_answer)

            errors.append({
                "index": i,
                "error": str(e),
            })

        elapsed = time.time() - start_time
        total_time += elapsed

        pred_answer.append(result)
        correct_answer.append(gt_answer_norm)

        print("Predicted:", result)
        print("Ground Truth:", gt_answer_norm)

    correct = 0
    total = len(pred_answer)

    for pred, true in zip(pred_answer, correct_answer):

        if str(pred).strip().lower() == str(true).strip().lower():
            correct += 1

    accuracy = correct / total
    avg_time = total_time / total

    all_accuracies.append(accuracy)
    all_avg_times.append(avg_time)
    all_total_tokens.append(total_tokens)

    print(f"\nRun {run_idx + 1} results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Avg time: {avg_time:.4f}")
    print(f"Total tokens: {total_tokens}")

In [ ]:
#независимые запуски
results = []

total_tokens = 0
total_time = 0

for i in tqdm(range(10)):

    question = df["problem"][i]
    true_answer = df["answer"][i]

    start_time = time.time()

    try:

        response = run_reasoning_system(question)

        answer = response["final_answer"]

        tokens = response.get("total_tokens", 0)

        total_tokens += tokens

        pred_norm = normalize_answer(answer)
        true_norm = normalize_answer(true_answer)

    except Exception as e:

        print(f"\nERROR at index {i}: {e}")

        answer = "ERROR"
        pred_norm = "ERROR"
        true_norm = normalize_answer(true_answer)

        response = {
            "error": str(e)
        }

        tokens = 0

    elapsed = time.time() - start_time

    total_time += elapsed

    is_correct = (
        str(pred_norm).strip().lower()
        ==
        str(true_norm).strip().lower()
    )

    if not is_correct:

        results.append({

            "index": i,

            "question": question,

            "true_answer": true_answer,
            "pred_answer": answer,

            "correct": is_correct,

            "tokens": tokens,
            "time_sec": elapsed,

            "error": response.get("error"),
        })

    print("Predicted:", answer)
    print("Ground Truth:", true_answer)
